# Phase 6 & 7: Model Training, Hyperparameter Tuning, and Stacking Ensemble

This notebook guides you through:
1. **Loading Preprocessed Splits**: Loading the resampled training set and clean test set saved in the previous SMOTE notebook.
2. **Hyperparameter Tuning**: Tuning Extra Trees and LightGBM using randomized search on a small balanced training subset.
3. **Base Model Training**: Fitting Random Forest, Extra Trees, and LightGBM on the full balanced dataset.
4. **Stacking Ensemble**: Combining predictions using a Logistic Regression meta-learner.
5. **Save Trained Models**: Saving the ensemble and individual classifiers using `joblib`.
6. **Evaluation**: Calculating F1-score, Precision, Recall, Accuracy, and saving Normalized Confusion Matrices.

## 1. Imports and Setup

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

# Add the parent directory to python path to import our src modules
sys.path.append(os.path.abspath('..'))

from src.config import (
    X_TRAIN_RESAMPLED_PATH,
    Y_TRAIN_RESAMPLED_PATH,
    X_TEST_PATH,
    Y_TEST_PATH,
    REPORTS_DIR,
    MODELS_DIR,
    RANDOM_STATE,
    ET_PARAM_GRID,
    LGBM_PARAM_GRID
)
from src.preprocessing import apply_resampling
from src.model_training import tune_hyperparameters, train_base_models, train_stacking_ensemble, save_model
from src.evaluation import evaluate_model, plot_confusion_matrix, save_comparison_report

print("All modules imported successfully!")

## 2. Load Split and Resampled Datasets
We load the resampled training dataset and the clean testing dataset from the disk.

In [ ]:
print("Loading resampled training set...")
X_train_final = pd.read_csv(X_TRAIN_RESAMPLED_PATH)
y_train_final = pd.read_csv(Y_TRAIN_RESAMPLED_PATH).iloc[:, 0]

print("Loading clean test set...")
X_test = pd.read_csv(X_TEST_PATH)
y_test = pd.read_csv(Y_TEST_PATH).iloc[:, 0]

print(f"Training set shape: {X_train_final.shape}")
print(f"Testing set shape: {X_test.shape}")
print("\nTraining class distribution:\n", y_train_final.value_counts())

## 3. Hyperparameter Tuning (Extra Trees and LightGBM)
We select a representative subset of the training data (e.g., 30,000 samples) to tune the hyperparameters of Extra Trees and LightGBM using `RandomizedSearchCV` quickly and efficiently without causing memory errors on your 16GB RAM laptop.

In [ ]:
from sklearn.model_selection import train_test_split

# Extract a stratified subset of 30,000 rows for hyperparameter tuning
X_train_sub, _, y_train_sub, _ = train_test_split(
    X_train_final,
    y_train_final,
    train_size=30000,
    stratify=y_train_final,
    random_state=RANDOM_STATE
)

print(f"Tuning subset shape: {X_train_sub.shape}")

# Tune Extra Trees
best_params_et = tune_hyperparameters(
    model_type="et",
    X_train_sub=X_train_sub,
    y_train_sub=y_train_sub,
    param_grid=ET_PARAM_GRID,
    n_iter=5
)

# Tune LightGBM
best_params_lgbm = tune_hyperparameters(
    model_type="lgbm",
    X_train_sub=X_train_sub,
    y_train_sub=y_train_sub,
    param_grid=LGBM_PARAM_GRID,
    n_iter=5
)

# Save best parameters to file
best_params = {"et": best_params_et, "lgbm": best_params_lgbm}
params_file = REPORTS_DIR / "best_params.json"
with open(params_file, "w") as f:
    json.dump(best_params, f, indent=4)
print(f"Optimized parameters cached to {params_file}")

## 4. Train Final Base Models
We train the Random Forest, Extra Trees, and LightGBM base models on the full resampled training dataset using their optimized parameters.

In [ ]:
# Load cached best parameters if skipping tuning cell
params_file = REPORTS_DIR / "best_params.json"
if params_file.exists():
    with open(params_file, "r") as f:
        best_params = json.load(f)
else:
    best_params = {"et": None, "lgbm": None}

base_models = train_base_models(
    X_train_final, 
    y_train_final, 
    best_params_et=best_params["et"], 
    best_params_lgbm=best_params["lgbm"]
)

## 5. Build and Train Stacking Ensemble
We combine the base classifiers using a Logistic Regression meta-learner. It utilizes 3-fold cross validation on base predictions to train the meta-learner safely without data leakage.

In [ ]:
stacking_model = train_stacking_ensemble(
    X_train_final, 
    y_train_final, 
    best_params_et=best_params["et"], 
    best_params_lgbm=best_params["lgbm"],
    meta_learner_type="logistic"
)

## 6. Save Trained Models
We persist the models to disk using `joblib` so they can be loaded by the FastAPI backend later.

In [ ]:
save_model(base_models["rf"], "rf_model.joblib")
save_model(base_models["et"], "et_model.joblib")
save_model(base_models["lgbm"], "lgbm_model.joblib")
save_model(stacking_model, "stacking_ensemble_model.joblib")
print("All models successfully saved to saved_models/!")

## 7. Evaluation and Plots
We generate classification reports for each classifier on the testing set and plot confusion matrices.

In [ ]:
results = []

# Evaluate Base Models
for name, model in base_models.items():
    y_pred = model.predict(X_test)
    metrics = evaluate_model(y_test, y_pred, f"{name.upper()} Model")
    plot_confusion_matrix(y_test, y_pred, f"{name.upper()} Model")
    results.append(metrics)

# Evaluate Stacking Ensemble
y_pred_stack = stacking_model.predict(X_test)
metrics_stack = evaluate_model(y_test, y_pred_stack, "Stacking Ensemble")
plot_confusion_matrix(y_test, y_pred_stack, "Stacking Ensemble")
results.append(metrics_stack)

# Save comparison reports
comparison_df = save_comparison_report(results, "model_comparison.csv")
comparison_df